# G5 · Síntesis final

**Spec:** [`docs/spec_G5_codex_final_synthesis.md`](../docs/spec_G5_codex_final_synthesis.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs12b_realigned`

Consolida G0–G4 en el paquete de caracterización (tablas + supuestos + síntesis).

| | |
|---|---|
| **Entrada** | G0–G4 |
| **Salida (QC/productos)** | `report/characterization/` (6 tablas + md + summary.json) |
| **Consume aguas abajo** | Revisión humana |


## Qué hace G5 y el cierre

G5 es la **síntesis final**: consolida G0–G4 en el paquete de caracterización (`report/characterization/`): tablas (parámetros adoptados, líneas, propiedades físicas, clasificación, índice de espectros, presupuesto de incertidumbre), documentos (`characterization.md`, `assumptions_and_limitations.md`) y **9 figuras**.

**Verifica (los V-checks):**
- **V1 trazabilidad:** hashes de QC de las fases G0–G4 (cadena completa).
- **V2 consistencia:** Hα de G2 frente al veredicto de E1 (`detected` vs `non_detection`); robustez de la clasificación: `secure`.
- **V4 determinismo:** hashes de los 10 archivos (dos builds → idénticos, reproducible).
- **V6 F1 intacto:** `run_summary_extended` (F1 no se toca, es aditivo).

**Resultado consolidado (ROXs 12 B):** clase = `companion_substellar_or_planetary` / `secure`; L_acc ≤ 5.74e-07 L☉; Ṁ ≲ 5.32e-14 (E3) — 1.83e-13 (G3).

**El G-block queda CERRADO (provisional):** el paquete está armado, es determinista y trazable, pero **hereda el bloqueo del A-block** (alineación, M3, M5) + el **tipado espectral diferido de G3** → G4 ambigua. Nada es paper-final hasta cerrar el A-block.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python scripts/build_characterization.py --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('report/characterization/characterization_summary.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python scripts/build_characterization.py --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('report/characterization/characterization_summary.json', RUN_ID)
nb.show(qc, keys=['final_class', 'consistency', 'provisional', 'n_lines', 'n_physical_properties'], title='G5')


## Los términos de este QC, en físico

G5 no añade ciencia: **empaqueta** y comprueba que lo que sale de la cadena es coherente consigo mismo.

| Término | Qué es | Por qué importa |
|---|---|---|
| `consistency` | Que las etapas no se contradigan entre sí (la Hα de E1/G2, el Ṁ de E3/G3, la clase de G4). | Dos caminos que miden lo mismo deben coincidir; si no, hay un factor aplicado de más o de menos. |
| `provisional` | Que algún ingrediente sigue abierto. | Un resultado provisional está calculado pero **no es publicable**: la etiqueta viaja con él para que no se cite por error. |
| `n_lines` / `n_physical_properties` | Cuántas líneas medidas y cuántas magnitudes físicas derivadas entran en el paquete. | Es el inventario de lo que realmente sostiene la síntesis. |
| `final_class` | La clasificación heredada de G4, con su ambigüedad si la tiene. | G5 no re-clasifica: transporta el veredicto y su incertidumbre. |


## Resultados que llevaron a la conclusión

Clase final, consistencia, determinismo y trazabilidad del `characterization_summary.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G5', 'report/characterization/characterization_summary.json'):
        q = nb.load_qc('report/characterization/characterization_summary.json', RUN_ID)
        fc = q['final_class']
        print(f"clase final: {fc['label']} / {fc['robustness']} ({fc['n_independent_supports']} soportes)")
        print('\nconsistencia:')
        for c in q['consistency']:
            print('  ', {k: v for k, v in c.items()})
        print(f"\npaquete: {q['n_lines']} líneas, {q['n_physical_properties']} propiedades físicas, "
              f"{len(q['figures_generated'])} figuras, {len(q['determinism_hash'])} archivos con hash (determinista)")
        print('trazabilidad (hashes de fase):', list(q['inputs']['phase_qc_hashes'].keys()))
        print('F1 intacto (V6):', q['f1_compatibility']['run_summary_extended'])
        oi = (q.get('open_issues') or [{}])[0]
        print('\nopen_issue (blocking):', (oi.get('issue') if isinstance(oi, dict) else str(oi))[:120])


## Plot 1 — el resultado consolidado del proyecto

La síntesis de toda la cadena A→G: qué es la fuente, y los límites de acreción. Con sus citas (de `adopted_parameters.csv`).


In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('report/characterization/characterization_summary.json', RUN_ID)
    ap = pd.read_csv(rd / 'report' / 'characterization' / 'adopted_parameters.csv').set_index('parameter')
    e3 = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
    e3_mdot = {L['method']: L['mdot'] for L in e3['limits']}[e3['canonical_method']]
    g3 = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
    ast = nb.load_qc('stages/stage01c_qc.json', RUN_ID)['astrometry']   # sep/PA oficiales (B3)
    MSUN_PER_MJUP = 1047.57
    m_mjup = float(ap.loc['companion_mass_msun', 'value']) * MSUN_PER_MJUP
    lines = [
        f"CLASE:  {q['final_class']['label']}  ({q['final_class']['robustness']})",
        f"        compañero real ligado a ROXs 12 A "
        f"(sep {ast['sep_arcsec']:.2f}\", PA {ast['pa_deg']:.0f}°)",
        '',
        f"ACRECIÓN (no-detección de Hα):",
        f"   L_acc  ≤ {g3['combined_accretion']['l_acc_lsun']:.1e} L☉      (Alcalá+2017 Hα)",
        f"   Ṁ      ≲ {e3_mdot:.1e} M☉/yr  (E3, Gumbel 99%)",
        f"   Ṁ      = {g3['mdot_p50_msun_yr']:.1e} M☉/yr  (G3, 5σ + R_in)",
        '',
        f"PARÁMETROS ADOPTADOS:",
        f"   distancia  {ap.loc['distance_pc','value']} pc   (Gaia)",
        f"   A_V        {ap.loc['a_v','value']}         (Rizzuto+2015)",
        f"   masa       {m_mjup:.1f} M_Jup    (Bowler+2017, hot-start)",
    ]
    fig, ax = plt.subplots(figsize=(8.5, 5)); ax.axis('off')
    ax.text(0.5, 0.98, f'G5 · Síntesis {nb.display_name(RUN_ID)} (provisional)',
            ha='center', va='top', fontsize=13, weight='bold')
    ax.text(0.05, 0.86, '\n'.join(lines), va='top', ha='left', fontsize=10, family='monospace')
    ax.text(0.5, 0.03, 'PROVISIONAL: caveats A-block (V2/V5/V6) + tipado espectral (G3) diferido -> subtipo ambiguo',
            ha='center', fontsize=8, color='tab:red')
    outdir = rd / 'plots' / 'g5_synthesis'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'summary_card.png', dpi=110, bbox_inches='tight'); print('figura ->', outdir / 'summary_card.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — integridad del paquete (los V-checks)

Lo que G5 verifica antes de cerrar: consistencia (Hα G2=H01), trazabilidad (hashes G0–G4), determinismo (dos builds → idénticos), F1 intacto. En rojo, el bloqueo heredado (A-block provisional).


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('report/characterization/characterization_summary.json', RUN_ID)
    hal = next((c for c in q['consistency'] if c.get('check') == 'halpha_category_g2_vs_h01'), {})
    checks = [
        ('V2 consistencia Hα (G2=H01)', bool(hal.get('consistent'))),
        (f"V1 trazabilidad (hashes G0–G4: {len(q['inputs']['phase_qc_hashes'])})", len(q['inputs']['phase_qc_hashes']) == 5),
        (f"V4 determinismo ({len(q['determinism_hash'])} archivos con hash)", len(q['determinism_hash']) >= 10),
        ('V6 F1 intacto (aditivo)', bool(q['f1_compatibility']['run_summary_extended'])),
        ('Cierre paper-final (caveats A-block V2/V5/V6 + tipado G3 pendientes)', False),
    ]
    names = [c[0] for c in checks]; oks = [c[1] for c in checks]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.barh(names, [1] * len(names), color=['tab:green' if o else 'tab:red' for o in oks])
    for i, (n, o) in enumerate(checks):
        ax.text(0.5, i, ('✓  ' if o else '✗  ') + n, ha='center', va='center', fontsize=9, color='w', weight='bold')
    ax.set_xlim(0, 1); ax.set_xticks([]); ax.set_yticks([]); ax.invert_yaxis()
    ax.set_title('G5 · integridad del paquete: determinista y trazable, provisional (caveats A-block + tipado G3)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g5_synthesis'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'integrity.png', dpi=110); print('figura ->', outdir / 'integrity.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **G-block CERRADO (provisional):** paquete completo (10 archivos con hash, 9 figuras) pero hereda el bloqueo del A-block + G3 diferido → clasificación ambigua.
- Determinismo verificado (dos builds → hash idéntico); trazabilidad (hashes G0–G4); F1 intacto (V6, aditivo).
- Consistencia V2: Hα G2 upper_limit = H01 non_detection.


## Conclusión (registrada) — cierre del proyecto

**G5: paquete de caracterización armado; determinista, trazable, F1 intacto; PROVISIONAL.**

- **Resultado consolidado:** el compañero de **ROXs 12 B** queda clasificado como `companion_substellar_or_planetary` con robustez `secure` (3 apoyos independientes); E1 = `non_detection` en Hα → **Ṁ ≲ 5.32e-14 M☉/yr** (E3) / 1.83e-13 (G3).
- **Integridad:** consistencia V2, trazabilidad G0–G4, determinismo (10 hashes), F1 intacto (V6).
- **Bloqueo heredado (al cierre, 2026-07-08):** A-block abierto (alineación, M3, M5) + tipado espectral de G3 diferido → nada era paper-final.
- **Actualización 2026-07-10 (ver A1):** los 6 blockers duros del A-block se **cerraron** (F1 realineado = yellow, 0 bloqueantes); quedan los caveats no bloqueantes V2/V5/V6 y el tipado espectral diferido → la validez para paper es juicio científico con esos caveats declarados.
- **Para cerrar del todo:** G3 real (librerías BT-Settl/BHAC15/Luhman-Bonnefoy) + 2ª época astrométrica romperían la ambigüedad.
